# The Carry Trade, Automated: An AI Workshop

**Idea:** borrow in a currency with low interest rates, invest in one with high rates, and keep the difference. That works as long as the exchange rate doesn't move against you.

This notebook builds the whole trading pipeline in four steps. In class, Claude writes each step's code live from the prompt shown above it. The code here is the reference version.

| Step | What it does |
|---|---|
| 1. Gather | Download exchange rates and interest rates |
| 2. Decide | Apply rules to pick the best trade |
| 3. Execute | Send the trades to a trading system (paper or OANDA practice) |
| 4. Evaluate | Backtest the rules and judge the decision |

**How to run:** *Runtime → Run all* in Colab, or *Run → Run All Cells* in Jupyter. If the live data download fails, the notebook switches to built-in practice data automatically.

In [ ]:
import io, os
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({"figure.figsize": (11, 5), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

## Settings: the class steers here

Every rule lives in this one cell. Change a number, then re-run everything below it.

In [ ]:
# ---- Data ----
USE_DEMO_DATA = False      # True = offline synthetic data (no internet needed)
START_DATE = "2005-01-01"
CURRENCIES = ["USD", "EUR", "JPY", "GBP", "CHF", "CAD", "AUD", "NZD", "NOK", "SEK"]

# ---- Trading rules (Step 2) ----
N_LONG = 3              # hold the N highest-rate currencies
N_SHORT = 3             # borrow in the N lowest-rate currencies
MIN_SPREAD_PCT = 1.0    # skip the trade if avg long rate - avg short rate is below this
VOL_TARGET = 0.08       # target annual volatility for position sizing (None = off)
VOL_LOOKBACK_MONTHS = 12

# ---- Execution (Step 3) ----
NOTIONAL_USD = 1_000_000
BROKER = "paper"        # "paper" or "oanda" (OANDA practice account)
SEND_ORDERS = False     # False = dry run: show the orders but send nothing

# ---- Evaluation (Step 4) ----
COST_BPS = 5            # trading cost per unit of turnover, in basis points

---
## Step 1: Gather exchange rates and interest rates

**Prompt for Claude:**
> Write Python that downloads, from FRED (free, no API key, via the fredgraph.csv URL), daily spot exchange rates vs the US dollar for EUR, JPY, GBP, CHF, CAD, AUD, NZD, NOK, SEK, and monthly 3-month interbank interest rates (the OECD IR3TIB01 series) for those countries plus the US, from 2005 onward. Convert every exchange rate to "US dollars per 1 unit of foreign currency", resample to month-end, and print a data-quality report.

**Watch for:** FRED quotes some pairs as *dollars per foreign unit* (EUR, GBP, AUD, NZD) and the rest as *foreign units per dollar*. Mix them up and every result comes out backwards. Did Claude catch it?

In [ ]:
# FRED series: (series id, how it is quoted)
FX_SERIES = {
    "EUR": ("DEXUSEU", "usd_per_fx"), "GBP": ("DEXUSUK", "usd_per_fx"),
    "AUD": ("DEXUSAL", "usd_per_fx"), "NZD": ("DEXUSNZ", "usd_per_fx"),
    "JPY": ("DEXJPUS", "fx_per_usd"), "CAD": ("DEXCAUS", "fx_per_usd"),
    "CHF": ("DEXSZUS", "fx_per_usd"), "NOK": ("DEXNOUS", "fx_per_usd"),
    "SEK": ("DEXSDUS", "fx_per_usd"),
}
RATE_SERIES = {
    "USD": "IR3TIB01USM156N", "EUR": "IR3TIB01EZM156N", "JPY": "IR3TIB01JPM156N",
    "GBP": "IR3TIB01GBM156N", "CHF": "IR3TIB01CHM156N", "CAD": "IR3TIB01CAM156N",
    "AUD": "IR3TIB01AUM156N", "NZD": "IR3TIB01NZM156N", "NOK": "IR3TIB01NOM156N",
    "SEK": "IR3TIB01SEM156N",
}

def fetch_fred(series_id):
    """Download one FRED series as a date-indexed Series."""
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text))
    s = pd.to_numeric(df[series_id], errors="coerce")       # FRED uses "." for missing
    s.index = pd.to_datetime(df[df.columns[0]])
    return s.dropna()

def gather_live():
    fx = {}
    for ccy, (sid, quote) in FX_SERIES.items():
        s = fetch_fred(sid)
        fx[ccy] = s if quote == "usd_per_fx" else 1.0 / s   # normalise the quote convention
    fx = pd.DataFrame(fx)
    fx["USD"] = 1.0
    fx = fx.resample("ME").last()
    rates = pd.DataFrame({c: fetch_fred(sid) for c, sid in RATE_SERIES.items()}).resample("ME").last()
    return fx[CURRENCIES], rates[CURRENCIES]

def gather_demo(seed=7):
    """Synthetic carry-like data for offline practice. Not real markets!"""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(START_DATE, "2026-09-30", freq="ME")
    base = {"USD": 2.5, "EUR": 1.5, "JPY": 0.1, "GBP": 2.8, "CHF": 0.3,
            "CAD": 2.4, "AUD": 4.0, "NZD": 4.3, "NOK": 2.9, "SEK": 1.4}
    crashes = pd.to_datetime(["2008-10-31", "2008-11-30", "2020-03-31", "2024-08-31"])
    rates, fx = {}, {}
    for ccy, level in base.items():
        r = np.empty(len(dates)); r[0] = level
        for i in range(1, len(dates)):
            r[i] = r[i-1] + 0.03 * (level - r[i-1]) + rng.normal(0, 0.15)
        rates[ccy] = np.clip(r, -0.75, None)
        if ccy == "USD":
            fx[ccy] = np.ones(len(dates)); continue
        beta = (level - 2.0) / 2.0                       # high yielders crash harder
        rets = rng.normal(0.0005 * beta, 0.025, len(dates))
        rets[dates.isin(crashes)] += -0.06 * beta
        fx[ccy] = np.exp(np.cumsum(rets)) * (0.01 if ccy == "JPY" else 1.0)
    return pd.DataFrame(fx, index=dates)[CURRENCIES], pd.DataFrame(rates, index=dates)[CURRENCIES]

In [ ]:
if USE_DEMO_DATA:
    fx, rates = gather_demo(); source = "DEMO (synthetic)"
else:
    try:
        fx, rates = gather_live(); source = "FRED (live)"
    except Exception as e:
        print(f"Live download failed ({e}). Switching to demo data.")
        fx, rates = gather_demo(); source = "DEMO (synthetic)"

fx, rates = fx.loc[START_DATE:], rates.loc[START_DATE:].ffill(limit=6)
common = fx.dropna(how="all").index.intersection(rates.dropna(how="all").index)
fx, rates = fx.loc[common], rates.loc[common]

print(f"Data source: {source}   |   {len(fx)} months, {fx.index[0]:%Y-%m} to {fx.index[-1]:%Y-%m}")

# Data-quality report
report = pd.DataFrame({
    "latest rate %": rates.iloc[-1],
    "rate last updated": [rates[c].last_valid_index().strftime("%Y-%m") for c in CURRENCIES],
    "1 USD buys": 1.0 / fx.iloc[-1],
}).sort_values("latest rate %", ascending=False)
report

In [ ]:
top3 = report.index[:3].tolist(); bottom3 = report.index[-3:].tolist()
ax = rates[top3 + bottom3].plot(lw=2, title="Interest rates: 3 highest vs 3 lowest today")
ax.set_ylabel("annual %"); plt.show()

---
## Step 2: Apply rules to pick the best trade

**Prompt for Claude:**
> Using this data, apply these rules each month: (1) rank currencies by interest rate; (2) go long the N highest and short the N lowest, equal weights; (3) skip the trade if the rate spread is below X points; (4) scale size to a target volatility using only past data. Show this month's target positions for a $1,000,000 account.

**Class votes:** N = 1 or 3? Minimum spread? Volatility sizing on or off? (Change the Settings cell.)

**Watch for:** rule 4 must use only past data (`.shift(1)`). Ask Claude: *"Does anything here use information we wouldn't have had at the time?"*

In [ ]:
def raw_weights(rates_row, n_long=N_LONG, n_short=N_SHORT, min_spread=MIN_SPREAD_PCT):
    """Rules 1-3 for one month: +1/n on longs, -1/n on shorts (sums to zero)."""
    r = rates_row.dropna().sort_values(ascending=False)
    w = pd.Series(0.0, index=rates_row.index)
    if len(r) < n_long + n_short:
        return w
    longs, shorts = r.index[:n_long], r.index[-n_short:]
    if r[longs].mean() - r[shorts].mean() < min_spread:
        return w                                   # not enough carry to be worth the risk
    w[longs], w[shorts] = 1.0 / n_long, -1.0 / n_short
    return w

def currency_returns(fx, rates):
    """Next-month return from holding each currency: carry (interest) + spot (FX move)."""
    carry = rates / 100 / 12
    spot = fx.pct_change().shift(-1)
    return carry, spot

def vol_scale(gross, target=VOL_TARGET, lookback=VOL_LOOKBACK_MONTHS):
    """Rule 4: size using PAST volatility only."""
    if target is None:
        return pd.Series(1.0, index=gross.index)
    realised = gross.rolling(lookback).std().shift(1) * np.sqrt(12)
    return (target / realised).clip(upper=2.0).fillna(1.0)

weights = rates.apply(raw_weights, axis=1)
carry, spot = currency_returns(fx, rates)
gross_hist = (weights * (carry + spot)).sum(axis=1).shift(1).dropna()
size = vol_scale(gross_hist).iloc[-1] if VOL_TARGET else 1.0

today = rates.index[-1]
w = weights.loc[today] * size
target = pd.DataFrame({"weight": w, "rate %": rates.loc[today], "usd_per_unit": fx.loc[today]})
target = target[target.weight != 0].sort_values("weight", ascending=False)
target["side"] = np.where(target.weight > 0, "LONG", "SHORT")
target["notional_usd"] = (target.weight * NOTIONAL_USD).round(0)

print(f"Signal date {today:%Y-%m-%d}   size multiplier {size:.2f}")
target[["side", "rate %", "notional_usd"]] if len(target) else print("No trade this month: spread too small.")

---
## Step 3: Send the trades to a trading system

**Prompt for Claude:**
> Convert each currency position into an FX order using OANDA instrument names (long AUD = buy AUD_USD; long JPY = sell USD_JPY), units in the base currency. Build a PaperBroker that records fills to a ledger and an OandaBroker for the v20 *practice* API, reading credentials from environment variables. Default to a dry run, and make it impossible to point at a live account.

**Discussion:** who should approve an order before it's sent, the code or a person?

To use OANDA: open a free practice account, set `BROKER = "oanda"`, and set the environment variables `OANDA_TOKEN` and `OANDA_ACCOUNT`.

In [ ]:
USD_QUOTED = {"EUR", "GBP", "AUD", "NZD"}       # OANDA quotes these as XXX_USD

def to_orders(target):
    rows = []
    for ccy, t in target.iterrows():
        if ccy == "USD":
            continue                              # USD legs net out automatically
        if ccy in USD_QUOTED:
            inst, units = f"{ccy}_USD", round(t.notional_usd / t.usd_per_unit)   # units in ccy
        else:
            inst, units = f"USD_{ccy}", round(-t.notional_usd)                   # units in USD
        rows.append({"instrument": inst, "units": int(units), "currency": ccy,
                     "side": t.side, "notional_usd": t.notional_usd})
    return pd.DataFrame(rows)

class PaperBroker:
    """Simulated broker: every order fills and is appended to ledger.csv."""
    def __init__(self, path="ledger.csv"):
        self.path = path
    def submit(self, orders):
        fills = orders.assign(timestamp=datetime.now(timezone.utc).isoformat(timespec="seconds"),
                              status="FILLED (paper)")
        fills.to_csv(self.path, mode="a", header=not os.path.exists(self.path), index=False)
        return fills

class OandaBroker:
    """OANDA v20 REST API -- practice environment only."""
    BASE_URL = "https://api-fxpractice.oanda.com"
    def __init__(self):
        assert "fxpractice" in self.BASE_URL, "Refusing to trade a live account."
        self.token, self.account = os.environ["OANDA_TOKEN"], os.environ["OANDA_ACCOUNT"]
    def submit(self, orders):
        url = f"{self.BASE_URL}/v3/accounts/{self.account}/orders"
        headers = {"Authorization": f"Bearer {self.token}", "Content-Type": "application/json"}
        out = []
        for _, o in orders.iterrows():
            body = {"order": {"type": "MARKET", "instrument": o.instrument, "units": str(o.units),
                              "timeInForce": "FOK", "positionFill": "DEFAULT"}}
            r = requests.post(url, json=body, headers=headers, timeout=30)
            ok = r.status_code == 201 and "orderFillTransaction" in r.json()
            out.append({**o.to_dict(), "status": "FILLED" if ok else f"REJECTED {r.status_code}"})
        return pd.DataFrame(out)

In [ ]:
orders = to_orders(target)
if SEND_ORDERS:
    broker = OandaBroker() if BROKER == "oanda" else PaperBroker()
    result = broker.submit(orders)
    print(f"Sent to {BROKER}:")
    display(result)
else:
    print("DRY RUN: nothing sent. Set SEND_ORDERS = True in Settings to submit.")
    display(orders)

---
## Step 4: Evaluate the decision

**Prompt for Claude:**
> Backtest the Step 2 rules month by month from 2005 to today. Split each month's return into interest income and exchange-rate moves, subtract a trading cost per unit of turnover, and report annual return, volatility, return per unit of risk, worst month and maximum drawdown, with and without volatility sizing. Chart growth of $1, with interest-only as a dashed line, and a drawdown panel.

In [ ]:
def backtest(use_vol_target=True, cost_bps=COST_BPS):
    carry_ret = (weights * carry).sum(axis=1)
    spot_ret = (weights * spot).sum(axis=1)
    scale = vol_scale(carry_ret + spot_ret) if (use_vol_target and VOL_TARGET) \
            else pd.Series(1.0, index=weights.index)
    w_scaled = weights.mul(scale, axis=0)
    turnover = w_scaled.diff().abs().sum(axis=1).fillna(w_scaled.abs().sum(axis=1))
    out = pd.DataFrame({"carry": carry_ret * scale, "spot": spot_ret * scale,
                        "costs": -turnover * cost_bps / 10_000})
    out["net"] = out.sum(axis=1)
    return out.iloc[:-1]                         # last month has no next-month return yet

def metrics(net):
    growth = (1 + net).cumprod()
    ann = growth.iloc[-1] ** (12 / len(net)) - 1
    vol = net.std() * np.sqrt(12)
    return {"annual return %": 100 * ann, "annual volatility %": 100 * vol,
            "return per unit risk": ann / vol, "worst month %": 100 * net.min(),
            "worst month": net.idxmin().strftime("%Y-%m"),
            "max drawdown %": 100 * (growth / growth.cummax() - 1).min()}

results = {"rules only": backtest(False), "rules + volatility sizing": backtest(True)}

base = results["rules only"]
print("Where the money came from (rules only, summed over the period):")
print(f"  interest income {100*base.carry.sum():7.1f}%\n  exchange rates  {100*base.spot.sum():7.1f}%"
      f"\n  trading costs   {100*base.costs.sum():7.1f}%")
pd.DataFrame({k: metrics(v.net) for k, v in results.items()})

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
for name, df in results.items():
    g = (1 + df.net).cumprod()
    ax1.plot(g.index, g, lw=2, label=name)
    ax2.plot(g.index, 100 * (g / g.cummax() - 1), lw=1.5)
ax1.plot(base.index, (1 + base.carry).cumprod(), ls="--", color="gray", lw=1,
         label="interest income only (no FX moves)")
ax1.set(ylabel="Growth of $1", title=f"Carry trade backtest ({source})"); ax1.legend(frameon=False)
ax2.set_ylabel("Drawdown %")
plt.tight_layout(); plt.show()

---
## Debrief

1. The gap between the dashed line and the strategy line is the exchange-rate risk you're paid to take. Was it worth it?
2. Look at the worst month. What happened in markets then? (Try 2008-10 and 2024-08.)
3. We picked these rules *after* seeing the history. Is that cheating? (Overfitting.)
4. Where did Claude's code need a human to check it?
5. What would you need before letting this system trade real money unsupervised?